# 20_deployment: executable lesson notebook

Read [the guide](README.md). These cells mirror every script and run offline by default.

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_hosting_decision

Source: `modules/20_deployment/examples/01_hosting_decision.py`.

In [ ]:
# Source: modules/20_deployment/examples/01_hosting_decision.py
"""Choose a host from workload requirements: Nimbus production lesson.

Run: uv run --all-groups python modules/20_deployment/examples/01_hosting_decision.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.production_lessons import choose_host


def main() -> None:
    lesson_settings()
    for scenario in [
        dict(seconds=5, streaming=False, managed_sessions=False, local=True),
        dict(seconds=25, streaming=False, managed_sessions=False, local=False),
        dict(seconds=1200, streaming=True, managed_sessions=False, local=False),
        dict(seconds=300, streaming=True, managed_sessions=True, local=False),
    ]:
        # KEY IDEA: session ownership and workload duration drive the choice.
        print(scenario, "->", choose_host(**scenario))
    complete()


run_lesson(main, "modules/20_deployment/examples/01_hosting_decision.py")

## 02_agentcore_local

Source: `modules/20_deployment/examples/02_agentcore_local.py`.

In [ ]:
# Source: modules/20_deployment/examples/02_agentcore_local.py
"""AgentCore entrypoint over local HTTP: Nimbus production lesson.

Run: uv run --all-groups python modules/20_deployment/examples/02_agentcore_local.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import asyncio

from common.deployment_apps import compare_local
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    # KEY IDEA: these are actual ASGI requests to the two app adapters, not AWS latency tests.
    print(asyncio.run(compare_local(cfg)))
    complete()


run_lesson(main, "modules/20_deployment/examples/02_agentcore_local.py")

## 03_agentcore_runtime

Source: `modules/20_deployment/examples/03_agentcore_runtime.py`.

In [ ]:
# Source: modules/20_deployment/examples/03_agentcore_runtime.py
"""Invoke Runtime with an isolated session: Nimbus production lesson.

Run: uv run --all-groups python modules/20_deployment/examples/03_agentcore_runtime.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.config import course_value
from common.deployment_calls import invoke_runtime
from common.lessons import complete
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    arn = (
        "arn:aws:bedrock-agentcore:us-east-1:111122223333:runtime/nimbus-AbCdEf1234"
        if cfg.provider == "mock"
        else course_value("AGENTCORE_RUNTIME_ARN")
    )
    # KEY IDEA: a new UUID isolates runtime hosting; conversation persistence is separate.
    print(invoke_runtime(cfg, arn))
    complete()


run_lesson(main, "modules/20_deployment/examples/03_agentcore_runtime.py")

## 04_lambda_handler

Source: `modules/20_deployment/examples/04_lambda_handler.py`.

In [ ]:
# Source: modules/20_deployment/examples/04_lambda_handler.py
"""The exact Lambda handler runs locally: Nimbus production lesson.

Run: uv run --all-groups python modules/20_deployment/examples/04_lambda_handler.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.production_support import lambda_handler


def main() -> None:
    _, cfg = lesson_settings()
    # KEY IDEA: the image uses this same handler, without credentials bundled into it.
    print(lambda_handler({"prompt": "Refund processing time?"}, None, settings=cfg))
    complete()


run_lesson(main, "modules/20_deployment/examples/04_lambda_handler.py")

## 05_fargate_api

Source: `modules/20_deployment/examples/05_fargate_api.py`.

In [ ]:
# Source: modules/20_deployment/examples/05_fargate_api.py
"""Authenticated FastAPI SSE adapter: Nimbus production lesson.

Run: uv run --all-groups python modules/20_deployment/examples/05_fargate_api.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import asyncio

from common.deployment_apps import compare_local
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    # KEY IDEA: these are actual ASGI requests to the two app adapters, not AWS latency tests.
    print(asyncio.run(compare_local(cfg)))
    complete()


run_lesson(main, "modules/20_deployment/examples/05_fargate_api.py")

## 06_runtime_configuration

Source: `modules/20_deployment/examples/06_runtime_configuration.py`.

In [ ]:
# Source: modules/20_deployment/examples/06_runtime_configuration.py
"""SSM configuration and Secrets Manager transport credentials: Nimbus production lesson.

Run: uv run --all-groups python modules/20_deployment/examples/06_runtime_configuration.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.config import course_value
from common.deployment_calls import runtime_configuration
from common.lessons import complete
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    parameter = (
        "/nimbus/service" if cfg.provider == "mock" else course_value("NIMBUS_SERVICE_PARAMETER")
    )
    secret = "nimbus-token" if cfg.provider == "mock" else course_value("NIMBUS_API_SECRET_ARN")
    # KEY IDEA: only a loaded/not-loaded flag leaves the credential boundary.
    print(runtime_configuration(cfg, parameter, secret))
    complete()


run_lesson(main, "modules/20_deployment/examples/06_runtime_configuration.py")

## deployment_comparison

Source: `modules/20_deployment/use_case/deployment_comparison.py`.

In [ ]:
# Source: modules/20_deployment/use_case/deployment_comparison.py
"""Compare identical support code on Lambda and Runtime: Nimbus production lesson.

Run: uv run --all-groups python modules/20_deployment/use_case/deployment_comparison.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import argparse
import asyncio

from common.config import course_value
from common.deployment_apps import compare_local
from common.deployment_calls import invoke_lambda, invoke_runtime
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--live-aws", action="store_true")
    args, cfg = lesson_settings(parser)
    if args.live_aws and args.mock:
        parser.error("Choose local --mock or --live-aws, not both")
    if args.live_aws:
        print("Lambda:", invoke_lambda(cfg, course_value("NIMBUS_LAMBDA_NAME")))
        print("Runtime:", invoke_runtime(cfg, course_value("AGENTCORE_RUNTIME_ARN")))
    else:
        print(asyncio.run(compare_local(cfg)))
    print("Repeat warm live measurements; include billed duration, tokens and idle resources.")
    complete()


run_lesson(main, "modules/20_deployment/use_case/deployment_comparison.py")